# Keyword Bid Optimizer

**Assignment**: Algorithm Developer (DS Specific) — Home Assignment  
**Goal**: Build a keyword bid optimization algorithm that maximizes ROAS while respecting budget and bid constraints.

---

## Table of Contents
1. [Data Exploration](#1-data-exploration)
2. [Part A — Bid Optimization Algorithm](#2-part-a--bid-optimization-algorithm)
3. [Part B — Handling Uncertainty & Low Data](#3-part-b--handling-uncertainty--low-data)
4. [Part C — Extension](#4-part-c--extension)
5. [Results & Validation](#5-results--validation)

## Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

# Constants from the assignment
BID_MIN = 0.20
BID_MAX = 15.00

---
## 1. Data Exploration

Before designing the algorithm, we need to understand the shape and quality of the data: how many keywords, how much data per keyword, the distribution of ROAS across campaigns, and how many keywords suffer from sparse data.

In [ ]:
df = pd.read_csv('campaign_data.csv')
df['date'] = pd.to_datetime(df['date'])
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

### 1.1 Data Quality Check

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")
print(f"Unique keywords: {df['keyword_id'].nunique()}")
print(f"Unique campaigns: {df['campaign_id'].nunique()}")
print(f"Unique match types: {df['match_type'].unique()}")

### 1.2 ROAS Distribution

In [ ]:
# Aggregate at keyword level (sum across all days)
kw = df.groupby(['keyword_id', 'keyword_text', 'campaign_id', 'campaign_name', 'match_type']).agg(
    total_spend=('spend', 'sum'),
    total_revenue=('revenue', 'sum'),
    total_clicks=('clicks', 'sum'),
    total_conversions=('conversions', 'sum'),
    total_impressions=('impressions', 'sum'),
    days_active=('date', 'nunique'),
    current_bid=('current_bid', 'last'),
    campaign_daily_budget=('campaign_daily_budget', 'last')
).reset_index()

kw['roas'] = kw['total_revenue'] / kw['total_spend'].replace(0, np.nan)
kw['cpc'] = kw['total_spend'] / kw['total_clicks'].replace(0, np.nan)
kw['cvr'] = kw['total_conversions'] / kw['total_clicks'].replace(0, np.nan)

print("ROAS stats:")
print(kw['roas'].describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

kw['roas'].clip(0, 20).hist(bins=40, ax=axes[0])
axes[0].set_title('ROAS Distribution (clipped at 20)')
axes[0].set_xlabel('ROAS')

kw['total_clicks'].hist(bins=40, ax=axes[1])
axes[1].set_title('Clicks per Keyword (30 days)')
axes[1].set_xlabel('Clicks')

kw['days_active'].hist(bins=30, ax=axes[2])
axes[2].set_title('Days Active per Keyword')
axes[2].set_xlabel('Days')

plt.tight_layout()
plt.show()

### 1.3 Campaign Budget Overview

In [ ]:
campaign_summary = df.groupby(['campaign_id', 'campaign_name']).agg(
    daily_budget=('campaign_daily_budget', 'last'),
    avg_daily_spend=('spend', 'mean'),
    keyword_count=('keyword_id', 'nunique')
).reset_index()
campaign_summary['budget_utilization'] = campaign_summary['avg_daily_spend'] / campaign_summary['daily_budget']
campaign_summary

---
## 2. Part A — Bid Optimization Algorithm

### 2.1 Problem Framing

**Objective**: For each keyword, recommend a new bid that:
- Maximizes ROAS across the portfolio
- Keeps total recommended daily spend per campaign ≤ campaign daily budget
- Keeps each bid in the range [$0.20, $15.00]

### 2.2 Core Intuition: ROAS-Proportional Bid Scaling

The fundamental insight is: **if a keyword is generating high ROAS, we want to spend more on it (raise the bid); if it is generating low ROAS, we want to spend less (lower the bid).**

The simplest formulation:
$$\text{new\_bid} = \text{current\_bid} \times \frac{\text{keyword\_ROAS}}{\text{target\_ROAS}}$$

Where `target_ROAS` is the campaign-level ROAS baseline (average ROAS of all keywords in the campaign).

This means:
- Keywords with ROAS **above** the campaign average → bid goes **up**
- Keywords with ROAS **below** the campaign average → bid goes **down**
- We scale proportionally, not with arbitrary multipliers

### 2.3 Budget Constraint

After computing scaled bids, we check if the estimated daily spend would exceed the campaign budget. If so, we apply a budget scaling factor:
$$\text{budget\_factor} = \frac{\text{campaign\_daily\_budget}}{\text{projected\_daily\_spend}}$$

All bids in that campaign are multiplied by this factor to bring the total spend within budget.

**Assumption**: We project daily spend as `new_bid × avg_daily_clicks_per_keyword`, summed across all keywords in the campaign.

### 2.4 Confidence Weighting (Low-Data Handling — Preview)

Keywords with very few clicks or days active have unreliable ROAS estimates. We introduce a **confidence weight** that shrinks the bid adjustment toward 0 (i.e., keep the current bid) when data is sparse. This is elaborated in Part B.

In [ ]:
class BidOptimizer:
    """
    Keyword bid optimizer using ROAS-proportional scaling with confidence weighting
    and campaign budget constraints.
    """

    BID_MIN = 0.20
    BID_MAX = 15.00

    # Low-data thresholds — keywords below these get conservative treatment
    MIN_CLICKS_FOR_FULL_CONFIDENCE = 30
    MIN_DAYS_FOR_FULL_CONFIDENCE = 7

    # Maximum bid change per optimization cycle (guardrail)
    MAX_BID_CHANGE_FACTOR = 2.0
    MIN_BID_CHANGE_FACTOR = 0.5

    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        self.df['date'] = pd.to_datetime(self.df['date'])

    def _aggregate(self) -> pd.DataFrame:
        """Aggregate daily rows to keyword-level statistics."""
        kw = self.df.groupby(
            ['keyword_id', 'keyword_text', 'campaign_id', 'match_type']
        ).agg(
            total_spend=('spend', 'sum'),
            total_revenue=('revenue', 'sum'),
            total_clicks=('clicks', 'sum'),
            total_conversions=('conversions', 'sum'),
            days_active=('date', 'nunique'),
            current_bid=('current_bid', 'last'),
            campaign_daily_budget=('campaign_daily_budget', 'last')
        ).reset_index()

        kw['roas'] = np.where(
            kw['total_spend'] > 0,
            kw['total_revenue'] / kw['total_spend'],
            np.nan
        )
        kw['avg_daily_clicks'] = kw['total_clicks'] / kw['days_active'].clip(lower=1)
        return kw

    def _compute_confidence(self, kw: pd.DataFrame) -> pd.Series:
        """
        Confidence score in [0, 1] based on click volume and days active.
        Score of 1 → full bid adjustment applied.
        Score of 0 → no bid adjustment (keep current bid).
        See Part B for detailed reasoning.
        """
        click_conf = (kw['total_clicks'] / self.MIN_CLICKS_FOR_FULL_CONFIDENCE).clip(0, 1)
        day_conf = (kw['days_active'] / self.MIN_DAYS_FOR_FULL_CONFIDENCE).clip(0, 1)
        return np.sqrt(click_conf * day_conf)  # geometric mean

    def _compute_target_roas(self, kw: pd.DataFrame) -> pd.Series:
        """Campaign-level ROAS benchmark (spend-weighted average)."""
        camp_roas = kw.groupby('campaign_id').apply(
            lambda g: g['total_revenue'].sum() / g['total_spend'].sum()
            if g['total_spend'].sum() > 0 else 1.0
        ).rename('target_roas')
        return kw['campaign_id'].map(camp_roas)

    def _apply_budget_constraint(self, kw: pd.DataFrame) -> pd.DataFrame:
        """
        Scale bids down within each campaign if projected daily spend
        exceeds the campaign daily budget.
        """
        kw = kw.copy()
        kw['projected_daily_spend'] = kw['recommended_bid'] * kw['avg_daily_clicks']

        camp_projected = kw.groupby('campaign_id')['projected_daily_spend'].transform('sum')
        budget = kw['campaign_daily_budget']

        budget_factor = np.where(
            camp_projected > budget,
            budget / camp_projected,
            1.0
        )
        kw['recommended_bid'] = (kw['recommended_bid'] * budget_factor).clip(
            self.BID_MIN, self.BID_MAX
        ).round(2)
        return kw

    def optimize(self) -> pd.DataFrame:
        """Run the full optimization pipeline and return bid recommendations."""
        kw = self._aggregate()

        # Confidence score for each keyword
        kw['confidence'] = self._compute_confidence(kw)

        # Campaign-level ROAS target
        kw['target_roas'] = self._compute_target_roas(kw)

        # ROAS-proportional scaling factor (how much to change the bid)
        roas_ratio = kw['roas'] / kw['target_roas'].replace(0, np.nan)

        # Clamp raw scaling to prevent extreme single-step jumps
        roas_ratio = roas_ratio.clip(
            self.MIN_BID_CHANGE_FACTOR, self.MAX_BID_CHANGE_FACTOR
        )

        # Blend scaled bid with current bid using confidence weight
        # confidence=1: use scaled bid fully
        # confidence=0: keep current bid unchanged
        scaled_bid = kw['current_bid'] * roas_ratio
        kw['recommended_bid'] = (
            kw['confidence'] * scaled_bid + (1 - kw['confidence']) * kw['current_bid']
        ).fillna(kw['current_bid'])  # fallback to current bid if ROAS is undefined

        # Hard bid bounds
        kw['recommended_bid'] = kw['recommended_bid'].clip(
            self.BID_MIN, self.BID_MAX
        ).round(2)

        # Budget constraint
        kw = self._apply_budget_constraint(kw)

        return kw[[
            'keyword_id', 'keyword_text', 'campaign_id', 'match_type',
            'current_bid', 'recommended_bid', 'roas', 'target_roas',
            'confidence', 'total_clicks', 'days_active'
        ]]

In [ ]:
# Run the optimizer
optimizer = BidOptimizer(df)
recommendations = optimizer.optimize()
recommendations.head(20)

### 2.5 Results — Bid Change Distribution

In [ ]:
recommendations['bid_change_pct'] = (
    (recommendations['recommended_bid'] - recommendations['current_bid'])
    / recommendations['current_bid'] * 100
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

recommendations['bid_change_pct'].hist(bins=40, ax=axes[0])
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Bid Change % Distribution')
axes[0].set_xlabel('Bid Change %')

axes[1].scatter(recommendations['roas'], recommendations['bid_change_pct'], alpha=0.4, s=20)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('ROAS')
axes[1].set_ylabel('Bid Change %')
axes[1].set_title('ROAS vs. Bid Change')

plt.tight_layout()
plt.show()

print(f"Keywords with bid increase: {(recommendations['bid_change_pct'] > 0).sum()}")
print(f"Keywords with bid decrease: {(recommendations['bid_change_pct'] < 0).sum()}")
print(f"Keywords unchanged: {(recommendations['bid_change_pct'] == 0).sum()}")

### 2.6 Budget Constraint Validation

In [ ]:
kw_full = optimizer._aggregate()
kw_full = kw_full.merge(
    recommendations[['keyword_id', 'match_type', 'recommended_bid']],
    on=['keyword_id', 'match_type']
)
kw_full['proj_daily_spend'] = kw_full['recommended_bid'] * kw_full['avg_daily_clicks']

budget_check = kw_full.groupby('campaign_id').agg(
    campaign_budget=('campaign_daily_budget', 'last'),
    projected_daily_spend=('proj_daily_spend', 'sum')
).reset_index()
budget_check['within_budget'] = budget_check['projected_daily_spend'] <= budget_check['campaign_budget']
budget_check['utilization'] = budget_check['projected_daily_spend'] / budget_check['campaign_budget']
print(budget_check.to_string(index=False))
print(f"\nAll campaigns within budget: {budget_check['within_budget'].all()}")

---
## 3. Part B — Handling Uncertainty & Low Data

### 3.1 The Problem

Not all keywords have the same amount of data. A keyword with 3 clicks in 30 days has a highly uncertain ROAS estimate — if those 3 clicks happened to generate one conversion, the ROAS looks amazing, but it could easily be noise. Conversely, a keyword with 500 clicks gives a stable, trustworthy ROAS.

**If we treat both keywords the same way, we risk over-investing in lucky low-data keywords and under-investing in genuinely good high-data keywords.**

### 3.2 What We Did: Confidence Weighting

We compute a `confidence` score ∈ [0, 1] for each keyword based on:
- **Click volume** — how many total clicks in the window (proxy for statistical reliability of ROAS)
- **Days active** — how many distinct days had data (proxy for recency spread / temporal stability)

$$\text{confidence} = \sqrt{\frac{\text{clicks}}{30} \times \frac{\text{days\_active}}{7}}$$

Both components are capped at 1, so a keyword with ≥30 clicks AND ≥7 days active gets full confidence = 1.

**Why geometric mean?** We want BOTH conditions to hold. A keyword with 200 clicks but only 1 day active is still risky (we may have had a promotion on that day). The geometric mean is 0 if either factor is 0, and only near 1 when both are strong.

### 3.3 How Confidence Affects Bids

The final bid is a weighted blend:
$$\text{new\_bid} = \text{confidence} \times \text{scaled\_bid} + (1 - \text{confidence}) \times \text{current\_bid}$$

- `confidence = 1`: We fully trust the ROAS signal → use the scaled bid
- `confidence = 0`: We have no reliable signal → keep the current bid unchanged
- `confidence = 0.5`: We partially trust the signal → blend halfway

This is a form of **Bayesian shrinkage** — pulling uncertain estimates toward a neutral prior (the current bid, which embeds historical decisions by the advertiser).

### 3.4 Additional Guardrails

Even for high-confidence keywords, we cap the bid change at **±50%** per optimization cycle (`MAX_BID_CHANGE_FACTOR = 2.0`, `MIN_BID_CHANGE_FACTOR = 0.5`). This prevents runaway bids from a single 30-day window of unusual performance.

In [ ]:
# Visualize confidence vs click count
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].scatter(
    recommendations['total_clicks'],
    recommendations['confidence'],
    alpha=0.4, s=20
)
axes[0].axvline(30, color='red', linestyle='--', label='30-click threshold')
axes[0].set_xlabel('Total Clicks (30 days)')
axes[0].set_ylabel('Confidence Score')
axes[0].set_title('Confidence vs. Click Volume')
axes[0].legend()

axes[1].scatter(
    recommendations['days_active'],
    recommendations['confidence'],
    alpha=0.4, s=20
)
axes[1].axvline(7, color='red', linestyle='--', label='7-day threshold')
axes[1].set_xlabel('Days Active')
axes[1].set_ylabel('Confidence Score')
axes[1].set_title('Confidence vs. Days Active')
axes[1].legend()

plt.tight_layout()
plt.show()

low_conf = recommendations[recommendations['confidence'] < 0.5]
print(f"Keywords with confidence < 0.5: {len(low_conf)} ({len(low_conf)/len(recommendations)*100:.1f}%)")

---
## 4. Part C — Extension

*(To be filled in after discussing extension options with the user.)*

**Options from the assignment:**
- Option 1: Match type differentiation (different bid logic per broad/phrase/exact)
- Option 2: Multi-metric optimization beyond ROAS (e.g., click-through rate, conversion rate)
- Option 3: Temporal trends — use recent days more heavily than older days

We will implement one of these here.

---
## 5. Results & Validation

### 5.1 Summary Statistics

In [ ]:
print("=== Bid Recommendation Summary ===")
print(f"Total keywords: {len(recommendations)}")
print(f"Average current bid: ${recommendations['current_bid'].mean():.2f}")
print(f"Average recommended bid: ${recommendations['recommended_bid'].mean():.2f}")
print(f"Average bid change: {recommendations['bid_change_pct'].mean():.1f}%")
print()
print("Bids at minimum ($0.20):", (recommendations['recommended_bid'] == 0.20).sum())
print("Bids at maximum ($15.00):", (recommendations['recommended_bid'] == 15.00).sum())

In [ ]:
# Save to CSV
recommendations.to_csv('bid_recommendations.csv', index=False)
print("Saved bid_recommendations.csv")